# AntMaze U-Maze qualification (binary NCE, modern stack)

Clean, self-contained qualification run for `antmaze_umaze` on the modern
Gymnasium-Robotics / MuJoCo / GPU-JAX stack. **No causal changes.**

Pipeline: pinned install -> clone+checkout tested commit -> mount Drive ->
**environment audit (stops on failure)** -> one-seed **100k-200k** qualification
(not 1M) with TensorBoard + init/early/mid/final/best/latest checkpoints +
NaN/saturation/fall/checkpoint guards + resume-from-latest -> `report_antmaze`
plots/video -> **PASS / WARN / FAIL** summary.

Leave `SMOKE = True` for a fast first pass, then set it `False` for the real run.

In [ ]:
# 1. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX.
#
# Colab ships a GPU-matched jax / jaxlib / numpy. ANY pip install that changes
# them -- directly, OR pulled in as a dependency of dm-haiku / optax / mujoco --
# hard-crashes the kernel on the next `import jax` (repeated 'kernel restarted',
# no traceback = native jaxlib/numpy mismatch). So we HOLD jax/jaxlib/numpy at
# Colab's exact versions while installing everything else. Run on a FRESH
# runtime (Runtime > Disconnect and delete runtime).
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}",
        f"numpy=={numpy.__version__}"]
print("Colab JAX", jax.__version__, "| devices:", jax.devices(), "| holding", hold)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium-robotics", "mujoco", "imageio-ffmpeg",
                "dm-haiku", "optax", *hold], check=True)
import os
os.environ["MUJOCO_GL"] = "egl"
print("post-install JAX", jax.__version__, "| devices:", jax.devices())
# If devices() shows CPU only, this runtime lacks a GPU -- switch to a GPU
# runtime (Runtime > Change runtime type > T4 GPU) and re-run.

In [ ]:
# 2. Clone the fork and checkout the tested commit.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "67cdade"   # tested commit for this notebook
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline

In [ ]:
# 3. Mount Google Drive (all checkpoints + artifacts saved here).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Parameters + guard-railed binary-NCE config.
import os, json
import numpy as np
from crl.config import Config
from crl.train import train
from crl import report_antmaze

ENV    = 'antmaze_umaze'
SEED   = 0
RESUME = False          # True to continue from latest.pkl
SMOKE  = True           # True = fast integration smoke; set False for the real run

if SMOKE:
    STEPS, MINREP, RANDS, EVAL, EVALEP, LOG = 4_000, 1_400, 1_400, 2_000, 2, 4_000
else:
    STEPS, MINREP, RANDS, EVAL, EVALEP, LOG = 150_000, 10_000, 10_000, 10_000, 10, 20_000

BASE    = '/content/drive/MyDrive/antmaze_umaze_qual'
RUN_DIR = f'{BASE}/{ENV}_s{SEED}' + ('_smoke' if SMOKE else '')
os.makedirs(RUN_DIR, exist_ok=True)

def build_config():
    cfg = Config(
        env_name=ENV, use_td=False, twin_q=False,        # binary NCE
        random_goals=0.5, entropy_coefficient=0.0,
        max_number_of_steps=STEPS,
        min_replay_size=MINREP, random_steps=RANDS,
        num_sgd_steps_per_step=4, batch_size=256,
        eval_every_steps=EVAL, eval_episodes=EVALEP, log_every_steps=LOG,
        seed=SEED, ckpt_dir=RUN_DIR, resume=RESUME, tensorboard=True,
    )
    assert cfg.use_td is False and cfg.twin_q is False, 'must be binary NCE'
    assert cfg.max_number_of_steps <= 200_000, 'qualification run, not 1M'
    return cfg

print(f'ENV={ENV} SEED={SEED} SMOKE={SMOKE} STEPS={STEPS} RESUME={RESUME}')
print('RUN_DIR:', RUN_DIR)

In [ ]:
# 5. Environment audit -- STOP before training if it fails.
ok, details = report_antmaze.env_audit(ENV)
print('AntMaze env audit:', 'PASS' if ok else 'FAIL')
for k, v in details.items():
    print(f'  {k}: {v}')
assert ok, 'AntMaze environment audit FAILED -- stopping before training.'

In [ ]:
# 6. Launch TensorBoard (critic/actor loss, logits gap, cat-acc, success,
#    distances, action saturation, torso height, fall fraction, goal velocity).
tb_logdir = RUN_DIR + '/tb'
%load_ext tensorboard
%tensorboard --logdir $tb_logdir

In [ ]:
# 7. Train (guard-railed). Milestones init/early/mid/final/best/latest are
#    saved to Drive; RESUME=True continues from latest.pkl. NaN guard in-loop.
cfg = build_config()
print('OK:', ENV, 'binary NCE, seed', SEED, '->', RUN_DIR)
state = train(cfg)

In [ ]:
# 8. Report + video + PASS/WARN/FAIL qualification summary.
from IPython.display import Image, Video, display
OUT  = f'{RUN_DIR}/report'
best = f'{RUN_DIR}/best.pkl'

rep = report_antmaze.full_report(ENV, ckpt=best, episodes=EVALEP, seed=123, out=OUT)
report_antmaze._print(rep)

# rendered rollout video (needs MUJOCO_GL=egl from cell 1)
vid = f'{OUT}/rollout.mp4'
try:
    report_antmaze.render_video(ENV, best, vid, episodes=1)
    display(Video(vid, embed=True, width=480))
except Exception as e:
    print('video skipped:', e)
display(Image(f'{OUT}/topdown_trajectories.png'))

verdict, issues, info = report_antmaze.qualification_verdict(
    f'{RUN_DIR}/metrics.json', RUN_DIR)
print('\n' + '=' * 52)
print('ANTMAZE QUALIFICATION:', verdict)
print('=' * 52)
for sev, msg in issues:
    print(f'  [{sev}] {msg}')
keep = ('success', 'action_saturation', 'fall_fraction', 'critic_loss_last',
        'logits_gap_last', 'cat_acc_last')
print('signals:', {k: (round(v, 3) if isinstance(v, float) else v)
                    for k, v in info.items() if k in keep})